<a href="https://colab.research.google.com/github/travistan101/linkedin-salary-predictor/blob/main/02_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML pipeline scope

This notebook focuses on:
- splitting data
- building ML pipelines
- baseline comparison
- agent-driven fine-tuning
- final model selection

## Why these machine learning models were selected

This project is a regression problem, where the target variable is `target_log_salary`.  
Three different regression algorithms were used to provide both a strong comparison and a meaningful tuning workflow:

1. **Linear Regression**
   - Used as a simple and interpretable baseline.
   - Provides a benchmark for how well a linear relationship can explain the target.

2. **Random Forest Regressor**
   - Used to capture non-linear relationships and feature interactions.
   - More flexible than Linear Regression and robust for structured tabular data.

3. **Spark XGBoost Regressor**
   - Used as a stronger boosted-tree model for tabular prediction.
   - Suitable for modelling more complex non-linear patterns and interactions across engineered features.

Using these three models allows comparison across increasing model complexity, from interpretable linear methods to more expressive tree-based ensemble methods.

# 1. Set up & Imports



In [ ]:
!pip install pyspark -q
!pip install langgraph
!pip install openai
!pip install langchain

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

FINAL_PARQUET_PATH = '/content/drive/MyDrive/BT4221 Group 13/Dataset/final_agent_handoff_parquet'

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np
import json

spark = SparkSession.builder.appName('SalaryPrediction').getOrCreate()
print(spark)


In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL = "gpt-4o-mini"

print("OpenAI client ready.")

OpenAI client ready.


# 2. Load data and features

In [ ]:
df = spark.read.parquet(FINAL_PARQUET_PATH)

print('Total rows:', df.count())

Total rows: 35321


# 3. Train / validation / test split

Creation of the final feature list that will be fed into the model

In [ ]:
label_col = "target_log_salary"

numeric_features = [
    "benefit_count",
    "log_employee_count",
    "log_follower_count",
    "seniority_score",
    "tech_skill_count",
    "company_prestige",
    "manager_x_company_size",
    "description_len",
    "exp_level_ordinal",
]

binary_features = [
    "location_has_remote_word",
    "flag_senior", "flag_lead", "flag_manager", "flag_director",
    "flag_associate", "flag_engineer", "flag_data", "flag_sales",
    "flag_finance", "flag_healthcare", "flag_project_product",
    "flag_executive", "flag_intern",
    "desc_has_python", "desc_has_sql", "desc_has_cloud", "desc_has_ai_ml",
    "desc_has_management", "desc_has_degree", "desc_has_certification",
    "desc_has_java",
    "401(k)", "Medical insurance", "Vision insurance",
    "Disability insurance", "Dental insurance", "Tuition assistance",
    "Commuter benefits",
]

categorical_features_ohe = [
    "formatted_work_type",
    "application_type",
    "remote_allowed_filled",
    "company_size",
    "employee_count_bucket",
]

extra_cols = ['title_clean', 'description_clean', 'country', 'location_state']

final_df_cols = (
    [label_col]
    + extra_cols
    + numeric_features
    + binary_features
    + categorical_features_ohe
)

print("Total final cols:", len(final_df_cols))
print(final_df_cols)

all_structured_cols = numeric_features + binary_features + categorical_features_ohe

model_df = (
    df.select(label_col, *all_structured_cols, *extra_cols)
      .dropna(subset=[label_col])
)

print('Structured features :', len(all_structured_cols))
print('Model rows          :', model_df.count())

Total final cols: 48
['target_log_salary', 'title_clean', 'description_clean', 'country', 'location_state', 'benefit_count', 'log_employee_count', 'log_follower_count', 'seniority_score', 'tech_skill_count', 'company_prestige', 'manager_x_company_size', 'description_len', 'exp_level_ordinal', 'location_has_remote_word', 'flag_senior', 'flag_lead', 'flag_manager', 'flag_director', 'flag_associate', 'flag_engineer', 'flag_data', 'flag_sales', 'flag_finance', 'flag_healthcare', 'flag_project_product', 'flag_executive', 'flag_intern', 'desc_has_python', 'desc_has_sql', 'desc_has_cloud', 'desc_has_ai_ml', 'desc_has_management', 'desc_has_degree', 'desc_has_certification', 'desc_has_java', '401(k)', 'Medical insurance', 'Vision insurance', 'Disability insurance', 'Dental insurance', 'Tuition assistance', 'Commuter benefits', 'formatted_work_type', 'application_type', 'remote_allowed_filled', 'company_size', 'employee_count_bucket']
Structured features : 43
Model rows          : 35321


In [ ]:
def apply_target_encoding(train, val, test, col_name, label, smooth=100):
    """
    Smoothed target encoding: blend category mean with global mean.
    Smooth controls how much to pull rare categories toward the global mean.
    """
    global_mean = train.select(F.mean(label)).collect()[0][0]
    stats = (train.groupBy(col_name)
                  .agg(F.mean(label).alias('cat_mean'), F.count('*').alias('cat_n')))
    stats = stats.withColumn(
        f'{col_name}_te',
        ((F.col('cat_mean') * F.col('cat_n') + global_mean * smooth)
         / (F.col('cat_n') + smooth)).cast('double')
    ).select(col_name, f'{col_name}_te')

    train = train.join(stats, on=col_name, how='left').fillna({f'{col_name}_te': global_mean})
    val   = val.join(stats,   on=col_name, how='left').fillna({f'{col_name}_te': global_mean})
    test  = test.join(stats,  on=col_name, how='left').fillna({f'{col_name}_te': global_mean})
    return train, val, test

train_df, val_df, test_df = model_df.randomSplit([0.8, 0.10, 0.10], seed=42)

train_df, val_df, test_df = apply_target_encoding(
    train_df, val_df, test_df, col_name="country", label=label_col
)

train_df, val_df, test_df = apply_target_encoding(
    train_df, val_df, test_df, col_name="location_state", label=label_col
)

train_df = train_df.drop("country", "location_state")
val_df = val_df.drop("country", "location_state")
test_df = test_df.drop("country", "location_state")

# 4. Preprocessing Pipeline

Performing preprocessing
1. TFIDF on title and description
2. OHE on categoricals
3. Assemble numeric, binary, categorical, and text features into final feature vectors
4. Apply scaling for Linear Regression only

In [ ]:
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    Tokenizer,
    StopWordsRemover,
    HashingTF,
    IDF,
    StandardScaler
)

# Final numeric inputs (structured + engineered + ordinal + target-encoded country)
numeric_assembler_inputs = numeric_features + ['exp_level_ordinal', 'country_te', 'location_state_te']

# TF-IDF on title_clean
tokenizer    = Tokenizer(inputCol='title_clean', outputCol='title_tokens')
remover      = StopWordsRemover(inputCol='title_tokens', outputCol='title_tokens_clean')
hashing_tf   = HashingTF(inputCol='title_tokens_clean', outputCol='title_tf', numFeatures=1024)
idf          = IDF(inputCol='title_tf', outputCol='title_tfidf', minDocFreq=10)

# TF-IDF on description_clean
desc_tokenizer = Tokenizer(inputCol='description_clean', outputCol='desc_tokens')
desc_remover   = StopWordsRemover(inputCol='desc_tokens', outputCol='desc_tokens_clean')
desc_tf        = HashingTF(inputCol='desc_tokens_clean', outputCol='desc_tf', numFeatures=4096)
desc_idf       = IDF(inputCol='desc_tf', outputCol='desc_tfidf', minDocFreq=100)

# OHE for remaining categoricals
indexers = [
    StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep')
    for c in categorical_features_ohe
]
encoder = OneHotEncoder(
    inputCols=[f'{c}_idx' for c in categorical_features_ohe],
    outputCols=[f'{c}_ohe' for c in categorical_features_ohe],
    handleInvalid='keep',
)

# Assemble all into one vector
assembler_inputs = (
    numeric_assembler_inputs
    + binary_features
    + [f'{c}_ohe' for c in categorical_features_ohe]
    + ['title_tfidf']
    + ['desc_tfidf']
)
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol='features_raw',
    handleInvalid='keep',
)

# StandardScaler, used only for Linear Regression
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features_scaled',
    withMean=False,
    withStd=True,
)

preprocessing_pipeline = Pipeline(stages=
    indexers + [
        encoder,
        tokenizer,
        remover,
        hashing_tf,
        idf,
        desc_tokenizer,
        desc_remover,
        desc_tf,
        desc_idf,
        assembler,
        scaler
    ]
)
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_prepared = preprocessing_model.transform(train_df)
val_prepared   = preprocessing_model.transform(val_df)
test_prepared  = preprocessing_model.transform(test_df)

print('Pipeline fit done.')
print('Columns available:', [c for c in train_prepared.columns if 'feature' in c])

Pipeline fit done.
Columns available: ['features_raw', 'features_scaled']


# 5. Defining Shared LangGraph State


The MLAgentState class defines the shared state used across the agent workflow, storing datasets, model results, evaluation outputs, tuning progress, and final selection in a consistent structure. This enables smooth data passing between nodes and keeps the workflow structured.

In [ ]:
from typing import TypedDict, Dict, List, Any

class MLAgentState(TypedDict, total=False):
    # Datasets
    train_prepared: Any
    val_prepared: Any
    test_prepared: Any

    # Baseline Stage
    fitted_baselines: Dict[str, Any]
    baseline_metrics: Dict[str, Dict[str, Any]]
    baseline_hyperparams: Dict[str, Dict[str, Any]]
    model_registry: Dict[str, Dict[str, Any]]
    baseline_ranking: List[str]

    # Evaluation Stage
    selected_for_tuning: str
    evaluation_summary: str
    evaluation_decision: Dict[str, Any]
    primary_metric: str
    secondary_metrics: List[str]
    diagnostic_metrics: List[str]
    next_step: str
    tuning_priority: str

    # Fine-tuning Stage
    tuning_plan: Dict[str, Dict[str, List[Any]]]
    tuned_models: Dict[str, Any]
    tuned_metrics: Dict[str, Dict[str, Any]]

    # Tuning Loop Fields
    run_history: List[Dict[str, Any]]
    best_params: Dict[str, Any]
    best_rmse: float
    current_params: Dict[str, Any]
    iteration: int
    decision: str
    tuning_target_model: str
    tuning_reasoning: str
    tuning_config: Dict[str, Any]

    # Search Tracking
    search_space: Dict[str, List[Any]]
    tried_param_signatures: List[str]
    tried_param_log: List[Dict[str, Any]]

    # Final Selection
    final_best_model: str
    final_ranking: List[str]

# 6. Evaluation Helper

Reusable functions to evaluate model performance across models.
We are evaluating on:
1. RMSE
2. MAE
3. R²


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(labelCol=label_col, predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(labelCol=label_col, predictionCol='prediction', metricName='r2')


def evaluate_predictions(pred_df):
    return {
        "rmse": float(evaluator_rmse.evaluate(pred_df)),
        "mae": float(evaluator_mae.evaluate(pred_df)),
        "r2": float(evaluator_r2.evaluate(pred_df)),
    }


def package_metrics(model_name, train_metrics, val_metrics, test_metrics):
    return {
        "model_name": model_name,
        "train": train_metrics,
        "val": val_metrics,
        "test": test_metrics,
        "gap_rmse_train_val": round(val_metrics["rmse"] - train_metrics["rmse"], 4),
        "gap_r2_train_val": round(train_metrics["r2"] - val_metrics["r2"], 4),
    }


def fit_and_evaluate_model(model_name, estimator, train_prepared, val_prepared, test_prepared):
    fitted_model = estimator.fit(train_prepared)

    train_pred = fitted_model.transform(train_prepared)
    val_pred = fitted_model.transform(val_prepared)
    test_pred = fitted_model.transform(test_prepared)

    metrics = package_metrics(
        model_name=model_name,
        train_metrics=evaluate_predictions(train_pred),
        val_metrics=evaluate_predictions(val_pred),
        test_metrics=evaluate_predictions(test_pred),
    )

    return fitted_model, metrics


def print_metrics_table(metrics):
    line = "─" * 55

    print(line)
    print(f"  {metrics['model_name']}")
    print(line)
    print(f"  {'Split':<10}{'RMSE':>10}{'MAE':>10}{'R²':>10}")

    for split_name in ["train", "val", "test"]:
        split_label = split_name.capitalize()
        split_metrics = metrics[split_name]

        print(
            f"  {split_label:<10}"
            f"{split_metrics['rmse']:>10.4f}"
            f"{split_metrics['mae']:>10.4f}"
            f"{split_metrics['r2']:>10.4f}"
        )

    print()
    print(f"  RMSE gap (Val - Train): {metrics['gap_rmse_train_val']:.4f}")
    print(f"  R² gap (Train - Val):   {metrics['gap_r2_train_val']:.4f}")
    print()


# 7. Baseline Training

Baseline training and evaluation for Linear Regression, Random Forest, and XGBoost. The resulting models, metrics, and hyperparameters are stored in the shared agent state for later comparison and fine-tuning.

In [ ]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor

MODEL_CONFIGS = {
    "LinearRegression_baseline": {
        "display_name": "LinearRegression",
        "estimator_cls": LinearRegression,
        "fixed_args": {
            "featuresCol": "features_scaled",
            "labelCol": label_col,
        },
        "baseline_params": {
            "maxIter": 100,
            "regParam": 0.01,
            "elasticNetParam": 0.0,
        },
        "prompt_context": """
You are tuning Spark LinearRegression for salary prediction with a log-transformed target.

== METRIC DIRECTIONS ==
- val RMSE     → LOWER is better (your ONLY goal)
- val R²       → HIGHER is better (secondary diagnostic)
- overfit_gap  → LOWER is better (diagnostic only, never a target)

== TUNING LOGIC ==
STEP 1 — ESTABLISH BASELINE
- Run starting params, observe val RMSE and overfit_gap

STEP 2 — IF underfitting
- Try increasing maxIter
- If regularization is later added and model looks too constrained, reduce regParam

STEP 3 — IF overfitting
- Increase regParam
- Explore elasticNetParam carefully

== PRINCIPLES ==
- Change ONE param at a time
- Must complete minimum 5 runs before considering stopping
- If a change makes val RMSE worse → revert to best params and try next lever
- Only stop if val RMSE improves < 0.002 across last 3 consecutive runs AND minimum 5 runs completed
""",
    },

    "RandomForest_baseline": {
        "display_name": "RandomForest",
        "estimator_cls": RandomForestRegressor,
        "fixed_args": {
            "featuresCol": "features_raw",
            "labelCol": label_col,
            "seed": 42,
        },
        "baseline_params": {
            "numTrees": 40,
            "maxDepth": 5,
        },
        "prompt_context": """
You are tuning Spark RandomForestRegressor for salary prediction with a log-transformed target.

== METRIC DIRECTIONS ==
- val RMSE     → LOWER is better (your ONLY goal)
- val R²       → HIGHER is better (secondary diagnostic)
- overfit_gap  → LOWER is better (diagnostic only, never a target)

== TUNING LOGIC ==
STEP 1 — ESTABLISH BASELINE
- Run starting params, observe val RMSE and overfit_gap

STEP 2 — IF underfitting
- Increase numTrees first
- Then consider increasing maxDepth

STEP 3 — IF overfitting
- Reduce maxDepth or add stronger leaf constraints if exposed

== PRINCIPLES ==
- Change ONE param at a time
- Must complete minimum 5 runs before considering stopping
- If a change makes val RMSE worse → revert to best params and try next lever
- Only stop if val RMSE improves < 0.002 across last 3 consecutive runs AND minimum 5 runs completed
""",
    },

    "XGBoost_baseline": {
        "display_name": "XGBoost",
        "estimator_cls": SparkXGBRegressor,
        "fixed_args": {
            "features_col": "features_raw",
            "label_col": label_col,
            "n_workers": 1,
            "use_gpu": False,
        },
        "baseline_params": {
            "num_rounds": 200,
            "max_depth": 8,
            "learning_rate": 0.1,
            "subsample": 0.8,
            "col_sample_by_tree": 0.8,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
            "min_child_weight": 1,
            "gamma": 0,
        },
        "prompt_context": """
You are tuning XGBoost for salary prediction with a log-transformed target.

== METRIC DIRECTIONS ==
- val RMSE     → LOWER is better (your ONLY goal)
- val R²       → HIGHER is better (secondary diagnostic)
- overfit_gap  → LOWER is better (diagnostic only, never a target)

== TUNING LOGIC ==
STEP 1 — ESTABLISH BASELINE
- Run starting params, observe val RMSE and overfit_gap

STEP 2 — IF overfit_gap IS LARGE
- Try each lever in order, one at a time:
  1. Reduce max_depth
  2. Increase reg_lambda
  3. Increase min_child_weight
- If all fail, move on

STEP 3 — CHASE VAL RMSE
- Increase num_rounds
- Reduce learning_rate paired with more rounds

== PRINCIPLES ==
- Change ONE param at a time
- Must complete minimum 5 runs before considering stopping
- If a change makes val RMSE worse → revert to best params and try next lever
- Only stop if val RMSE improves < 0.002 across last 3 consecutive runs AND minimum 5 runs completed
""",
    },
}

In [ ]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor

def build_estimator(model_name: str, params: dict):
    cfg = MODEL_CONFIGS[model_name]
    return cfg["estimator_cls"](
        **cfg["fixed_args"],
        **params,
    )

BASELINE_HYPERPARAMS = {
    model_name: cfg["baseline_params"].copy()
    for model_name, cfg in MODEL_CONFIGS.items()
}

state: MLAgentState = {
    "train_prepared": train_prepared,
    "val_prepared": val_prepared,
    "test_prepared": test_prepared,
}

def train_baseline_models(state: MLAgentState) -> MLAgentState:
    train_prepared = state["train_prepared"]
    val_prepared = state["val_prepared"]
    test_prepared = state["test_prepared"]

    fitted_baselines = {}
    baseline_metrics = {}
    baseline_hyperparams = {}

    print("Starting baseline model training...\n")

    for model_name, cfg in MODEL_CONFIGS.items():
        params = cfg["baseline_params"].copy()
        estimator = build_estimator(model_name, params)

        print(f"Training {model_name}...\n")

        fitted_model, metrics = fit_and_evaluate_model(
            model_name=model_name,
            estimator=estimator,
            train_prepared=train_prepared,
            val_prepared=val_prepared,
            test_prepared=test_prepared,
        )

        fitted_baselines[model_name] = fitted_model
        baseline_metrics[model_name] = metrics
        baseline_hyperparams[model_name] = params

        print_metrics_table(metrics)

    state["fitted_baselines"] = fitted_baselines
    state["baseline_metrics"] = baseline_metrics
    state["baseline_hyperparams"] = baseline_hyperparams

    print("Baseline model training completed.\n")
    return state

state = train_baseline_models(state)

Starting baseline model training...

Training LinearRegression_baseline...

───────────────────────────────────────────────────────
  LinearRegression_baseline
───────────────────────────────────────────────────────
  Split           RMSE       MAE        R²
  Train         0.3010    0.2104    0.7196
  Val           0.4230    0.2696    0.4887
  Test          0.3411    0.2515    0.6247

  RMSE gap (Val - Train): 0.1220
  R² gap (Train - Val):   0.2309

Training RandomForest_baseline...

───────────────────────────────────────────────────────
  RandomForest_baseline
───────────────────────────────────────────────────────
  Split           RMSE       MAE        R²
  Train         0.4282    0.3225    0.4328
  Val           0.4799    0.3333    0.3418
  Test          0.4206    0.3189    0.4293

  RMSE gap (Val - Train): 0.0518
  R² gap (Train - Val):   0.0909

Training XGBoost_baseline...



INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.8, 'n_workers': 1, 'use_gpu': False, 'num_rounds': 200, 'col_sample_by_tree': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


───────────────────────────────────────────────────────
  XGBoost_baseline
───────────────────────────────────────────────────────
  Split           RMSE       MAE        R²
  Train         0.2354    0.1754    0.8285
  Val           0.3993    0.2373    0.5444
  Test          0.3021    0.2214    0.7056

  RMSE gap (Val - Train): 0.1639
  R² gap (Train - Val):   0.2841

Baseline model training completed.



# 8. Baseline Evaluation & Selecting Best Model for Tuning
Purpose:
- rank baseline models using validation metrics
- select one best model for tuning
- store decision + reasoning in shared state

In [ ]:
# Function to rank models
def baseline_evaluation_node(state: MLAgentState) -> MLAgentState:
    baseline_metrics = state["baseline_metrics"]

    if not baseline_metrics:
        raise ValueError("baseline_metrics is empty. Run baseline training first.")

    # Rank models:
    # 1) lowest validation RMSE
    # 2) lower validation MAE
    # 3) higher validation R²
    ranked = sorted(
        baseline_metrics.items(),
        key=lambda x: (
            x[1]["val"]["rmse"],
            x[1]["val"]["mae"],
            -x[1]["val"]["r2"]
        )
    )

    best_model_name, best_metrics = ranked[0]

    gap_rmse = best_metrics["gap_rmse_train_val"]
    gap_r2 = best_metrics["gap_r2_train_val"]

    overfit_flag = gap_rmse > 0.12
    high_generalization_gap = gap_r2 > 0.20

    if overfit_flag or high_generalization_gap:
        tuning_priority = "reduce_overfitting_first_then_improve_val_rmse"
        next_step = "tune_selected_model_with_regularization_focus"
    else:
        tuning_priority = "improve_val_rmse_then_local_refinement"
        next_step = "tune_selected_model_for_rmse"

    summary_lines = []
    summary_lines.append("Baseline evaluation summary")
    summary_lines.append("")

    for rank, (model_name, metrics) in enumerate(ranked, start=1):
        summary_lines.append(
            f"{rank}. {model_name} | "
            f"Val RMSE={metrics['val']['rmse']:.4f}, "
            f"Val MAE={metrics['val']['mae']:.4f}, "
            f"Val R²={metrics['val']['r2']:.4f}, "
            f"RMSE gap={metrics['gap_rmse_train_val']:.4f}, "
            f"R² gap={metrics['gap_r2_train_val']:.4f}"
        )

    summary_lines.append("")
    summary_lines.append(f"Selected baseline for tuning: {best_model_name}")
    summary_lines.append(
        f"Selection reason: lowest validation RMSE = {best_metrics['val']['rmse']:.4f}"
    )

    if overfit_flag or high_generalization_gap:
        summary_lines.append(
            "Diagnostic: train-validation gap is noticeable, so the next stage should focus on controlling overfitting first."
        )
    else:
        summary_lines.append(
            "Diagnostic: generalization looks acceptable, so the next stage can focus mainly on improving validation RMSE."
        )

    # Write structured decision into state
    state["primary_metric"] = "val_rmse"
    state["secondary_metrics"] = ["val_mae", "val_r2"]
    state["diagnostic_metrics"] = ["gap_rmse_train_val", "gap_r2_train_val"]

    state["baseline_ranking"] = [name for name, _ in ranked]
    state["selected_for_tuning"] = best_model_name
    state["next_step"] = next_step
    state["tuning_priority"] = tuning_priority

    state["evaluation_summary"] = "\n".join(summary_lines)
    state["evaluation_decision"] = {
        "baseline_ranking": [name for name, _ in ranked],
        "selected_for_tuning": best_model_name,
        "primary_metric": "val_rmse",
        "secondary_metrics": ["val_mae", "val_r2"],
        "diagnostic_metrics": ["gap_rmse_train_val", "gap_r2_train_val"],
        "selected_val_rmse": best_metrics["val"]["rmse"],
        "selected_val_mae": best_metrics["val"]["mae"],
        "selected_val_r2": best_metrics["val"]["r2"],
        "gap_rmse_train_val": gap_rmse,
        "gap_r2_train_val": gap_r2,
        "overfit_flag": overfit_flag,
        "high_generalization_gap": high_generalization_gap,
        "tuning_priority": tuning_priority,
        "next_step": next_step,
        "selection_rule": "Lowest validation RMSE, then lower validation MAE, then higher validation R2"
    }

    return state

state = baseline_evaluation_node(state)

print(state["evaluation_summary"])
print()
print("Selected for tuning:", state["selected_for_tuning"])
print("Next step:", state["next_step"])

Baseline evaluation summary

1. XGBoost_baseline | Val RMSE=0.3993, Val MAE=0.2373, Val R²=0.5444, RMSE gap=0.1639, R² gap=0.2841
2. LinearRegression_baseline | Val RMSE=0.4230, Val MAE=0.2696, Val R²=0.4887, RMSE gap=0.1220, R² gap=0.2309
3. RandomForest_baseline | Val RMSE=0.4799, Val MAE=0.3333, Val R²=0.3418, RMSE gap=0.0518, R² gap=0.0909

Selected baseline for tuning: XGBoost_baseline
Selection reason: lowest validation RMSE = 0.3993
Diagnostic: train-validation gap is noticeable, so the next stage should focus on controlling overfitting first.

Selected for tuning: XGBoost_baseline
Next step: tune_selected_model_with_regularization_focus


# 9. Fine-tuning agent
Purpose:
- Read the selected baseline model from state
- Pull model-specific metadata from the registry
- Pull baseline hyperparameters from state as the starting point
- Run the same fine-tuning loop for LinearRegression, RandomForest, or XGBoost


In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END

def train_and_evaluate_candidate(
    model_name: str,
    params: dict,
    train_prepared,
    val_prepared,
    test_prepared,
):
    estimator = build_estimator(model_name, params)

    fitted_model, metrics = fit_and_evaluate_model(
        model_name=f"{model_name}_tuned_candidate",
        estimator=estimator,
        train_prepared=train_prepared,
        val_prepared=val_prepared,
        test_prepared=test_prepared,
    )

    return {
        "model_name": model_name,
        "params": params,
        "train_rmse": round(metrics["train"]["rmse"], 4),
        "val_rmse": round(metrics["val"]["rmse"], 4),
        "test_rmse": round(metrics["test"]["rmse"], 4),
        "val_r2": round(metrics["val"]["r2"], 4),
        "overfit_gap": round(metrics["gap_rmse_train_val"], 4),
        "full_metrics": metrics,
        "fitted_model": fitted_model,
    }

In [ ]:
import json

## Defining search spaces, parameters tuning priority order for each model
SEARCH_SPACES = {
    "LinearRegression_baseline": {
        "maxIter": [50, 100, 150, 200, 300],
        "regParam": [0.0, 0.001, 0.01, 0.05, 0.1],
        "elasticNetParam": [0.0, 0.2, 0.5, 0.8, 1.0],
    },
    "RandomForest_baseline": {
        "numTrees": [20, 40, 80, 120, 200],
        "maxDepth": [3, 5, 7, 9, 12],
    },
    "XGBoost_baseline": {
        "num_rounds": [100, 150, 200, 300, 400],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.03, 0.05, 0.1, 0.15],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "col_sample_by_tree": [0.7, 0.8, 0.9, 1.0],
        "reg_lambda": [0.5, 1.0, 2.0, 5.0],
        "reg_alpha": [0.0, 0.1, 0.5, 1.0],
        "min_child_weight": [1, 3, 5, 7],
        "gamma": [0.0, 0.1, 0.3, 0.5],
    },
}

PRIORITY_ORDER = {
    "LinearRegression_baseline": [
        "regParam",
        "elasticNetParam",
        "maxIter",
    ],
    "RandomForest_baseline": [
        "numTrees",
        "maxDepth",
    ],
    "XGBoost_baseline": [
        "max_depth",
        "min_child_weight",
        "reg_lambda",
        "gamma",
        "learning_rate",
        "num_rounds",
        "subsample",
        "col_sample_by_tree",
        "reg_alpha",
    ],
}


def normalize_param_value(v):
    if isinstance(v, float):
        return round(v, 8)
    return v


def param_signature(params: dict) -> str:
    normalized = {
        k: normalize_param_value(v)
        for k, v in sorted(params.items())
    }
    return json.dumps(normalized, sort_keys=True)


def adjacent_values(space_values, current_value):
    values = list(space_values)

    if current_value not in values:
        return values

    idx = values.index(current_value)
    ordered = []

    for step in range(1, len(values)):
        left = idx - step
        right = idx + step

        if left >= 0:
            ordered.append(values[left])
        if right < len(values):
            ordered.append(values[right])

    return ordered


def generate_candidates(
    model_name: str,
    anchor_params: dict,
    tried_param_signatures: list[str],
    max_candidates: int = 10,
):
    spaces = SEARCH_SPACES[model_name]
    priority_order = PRIORITY_ORDER[model_name]
    tried = set(tried_param_signatures)

    candidates = []

    for param in priority_order:
        current_value = anchor_params[param]

        for candidate_value in adjacent_values(spaces[param], current_value):
            candidate_params = anchor_params.copy()
            candidate_params[param] = candidate_value

            sig = param_signature(candidate_params)

            if sig not in tried:
                candidates.append({
                    "changed_param": param,
                    "params": candidate_params,
                })

            if len(candidates) >= max_candidates:
                return candidates

    return candidates

#10. Agent Skills
Purpose:
- Stateless, reusable functions that encapsulate a single capability.
- Called by graph nodes which are responsible for unpacking state.


Skills :
1.   select_candidate: Pick the next hyperparameter candidate to try from the allowed options.
2.   tuning_decision: Deciding whether tuning should continue or stop, while enforcing guardrails like minimum run count and search-space exhaustion



In [ ]:
## Defining system prompt, ensuring it comes back in valid JSON
SYSTEM = "You are a model hyperparameter tuning expert. Respond in valid JSON only."

def ask_llm(model_name: str, starting_params: dict, prompt: str) -> dict:
    cfg = MODEL_CONFIGS[model_name]

    context = f"""
{cfg['prompt_context']}

== MODEL ==
{model_name}

== FIXED EXECUTION SETTINGS ==
{json.dumps(cfg['fixed_args'])}

== STARTING PARAMS ==
{json.dumps(starting_params)}

== METRIC DIRECTIONS ==
- val RMSE -> LOWER is better (primary objective)
- val R² -> HIGHER is better (secondary diagnostic)
- overfit_gap -> LOWER is better (diagnostic only)
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": context + "\n\n" + prompt},
        ],
    )
    content = response.choices[0].message.content.strip()
    content = content.replace("```json", "").replace("```", "").strip()
    return json.loads(content)



# -----------------------------
# SKILL 1: Candidate Selection
# -----------------------------

def select_candidate(
    model_name: str,
    starting_params: dict,
    anchor_params: dict,
    candidates: list,
    run_history: list,
    best_rmse: float,
    iteration: int,
) -> dict:
    """
    Pick the best candidate to try next.
    Returns: {"candidate_id": int, "chosen": dict, "reasoning": str}
    """
    run_history_summary = [
        {
            "iteration": r["iteration"],
            "params": r["params"],
            "val_rmse": r["val_rmse"],
            "val_r2": r["val_r2"],
            "overfit_gap": r["overfit_gap"],
        }
        for r in run_history
    ]

    candidate_payload = [
        {
            "candidate_id": i,
            "changed_param": c["changed_param"],
            "params": c["params"],
        }
        for i, c in enumerate(candidates)
    ]

    prompt = f"""
Current best val RMSE: {best_rmse}
Current iteration: {iteration}

Anchor params (current best):
{json.dumps(anchor_params, indent=2)}

Run history:
{json.dumps(run_history_summary, indent=2)}

You MUST choose exactly one candidate_id from the allowed candidates below.
Do NOT invent new params.
Do NOT repeat already tried combinations.
Prefer candidates that are close to the current best and make sense based on the run history.

Allowed candidates:
{json.dumps(candidate_payload, indent=2)}

Return valid JSON in exactly this structure:
{{
  "candidate_id": 0,
  "reasoning": "..."
}}
"""

    try:
        out = ask_llm(model_name, starting_params, prompt)
        chosen_id = int(out["candidate_id"])
        return {
            "candidate_id": chosen_id,
            "chosen": candidates[chosen_id],
            "reasoning": out["reasoning"],
        }
    except Exception as e:
        return {
            "candidate_id": 0,
            "chosen": candidates[0],
            "reasoning": f"Fallback: planner parsing failed: {str(e)}",
        }


# -------------------------
# SKILL 2: Tuning Decision
# -------------------------

def tuning_decision(
    model_name: str,
    starting_params: dict,
    run_history: list,
    best_rmse: float,
    iteration: int,
    remaining_candidate_count: int,
) -> dict:
    """
    Decide whether to continue or stop tuning.
    Applies hard guardrails after LLM response:
    - < 5 iterations → forced continue
    - 0 remaining candidates → forced done
    Returns: {"decision": str, "reasoning": str}
    """
    run_history_summary = [
        {
            "iteration": r["iteration"],
            "params": r["params"],
            "val_rmse": r["val_rmse"],
            "val_r2": r["val_r2"],
            "overfit_gap": r["overfit_gap"],
        }
        for r in run_history
    ]

    prompt = f"""
Run history:
{json.dumps(run_history_summary, indent=2)}

Best val RMSE: {best_rmse}
Iteration: {iteration}/15
Remaining candidate count: {remaining_candidate_count}

Stop only if:
- minimum 5 runs have completed, and
- improvement is very small across recent runs,
  OR no untried candidates remain.

Return valid JSON in exactly one of these structures:

{{
  "decision": "continue",
  "reasoning": "..."
}}

OR

{{
  "decision": "done",
  "reasoning": "..."
}}
"""

    try:
        out = ask_llm(model_name, starting_params, prompt)
    except Exception as e:
        out = {
            "decision": "done",
            "reasoning": f"Fallback: LLM parsing failed: {str(e)}",
        }

    # Guardrail: minimum 5 runs
    if iteration < 5 and out["decision"] == "done":
        out["decision"] = "continue"
        out["reasoning"] += " | Minimum 5 runs not reached."

    # Guardrail: search space exhausted
    if remaining_candidate_count == 0:
        out["decision"] = "done"
        out["reasoning"] = out.get("reasoning", "") + " | Search space exhausted."

    return out

#11. LangGraph nodes and agentic workflow
Purpose:
- Defines the 4-node graph execution loop.
- Flow: init -> planner -> training -> analyser -> router -> loop/end
- Each node updates MLAgentState to track tuning progress and decisions.

Init node

Sets up the tuning process by loading the selected model, baseline parameters, search space, and tracking fields into the shared state.

In [ ]:
def fine_tuning_agent_init_node(state: MLAgentState) -> MLAgentState:
    selected_model = state["selected_for_tuning"]

    if selected_model not in MODEL_CONFIGS:
        raise ValueError(f"selected_for_tuning not found in MODEL_CONFIGS: {selected_model}")

    baseline_params = state["baseline_hyperparams"][selected_model].copy()

    state["tuning_target_model"] = selected_model
    state["run_history"] = []
    state["best_params"] = baseline_params.copy()
    state["best_rmse"] = float("inf")
    state["current_params"] = baseline_params.copy()
    state["iteration"] = 0
    state["decision"] = "continue"
    state["tuning_reasoning"] = f"Initialized tuning for {selected_model}"
    state["tuned_models"] = state.get("tuned_models", {})
    state["tuned_metrics"] = state.get("tuned_metrics", {})
    state["search_space"] = SEARCH_SPACES[selected_model]
    state["tried_param_signatures"] = []
    state["tried_param_log"] = []

    return state

Planner node

Chooses the next hyperparameter combination to try, starting with the baseline first and then selecting new candidates based on past results.

In [ ]:
def planner(state: MLAgentState):
    model_name = state["tuning_target_model"]
    starting_params = state["baseline_hyperparams"][model_name]

    # First run = baseline params
    if state["iteration"] == 0:
        print("[Planner] First run uses baseline params.")
        print(f"[Planner] params={starting_params}")

        return {
            "current_params": starting_params,
            "decision": "continue",
            "tuning_reasoning": "First run uses the baseline parameter combination.",
        }

    anchor_params = state["best_params"].copy()

    candidates = generate_candidates(
        model_name=model_name,
        anchor_params=anchor_params,
        tried_param_signatures=state.get("tried_param_signatures", []),
        max_candidates=8,
    )

    if not candidates:
        print("[Planner] No untried candidates left.")
        return {
            "decision": "done",
            "tuning_reasoning": "Search space exhausted. No untried candidate combinations remain.",
        }

    result = select_candidate(
        model_name=model_name,
        starting_params=starting_params,
        anchor_params=anchor_params,
        candidates=candidates,
        run_history=state["run_history"],
        best_rmse=state["best_rmse"],
        iteration=state["iteration"],
    )

    chosen = result["chosen"]
    print(f"[Planner] {result['reasoning']}")
    print(f"[Planner] changed {chosen['changed_param']} -> {chosen['params'][chosen['changed_param']]}")
    print(f"[Planner] params={chosen['params']}")

    return {
        "current_params": chosen["params"],
        "decision": "continue",
        "tuning_reasoning": result["reasoning"],
    }

Training node

Trains the model using the chosen parameters, evaluates its performance, and updates the tuning history and current best result.

In [ ]:
def train(state: MLAgentState):
    # Guardrail: no more candidates to try
    if state.get("decision") == "done":
        print("Train skipped — no more candidates to try")
        return {}

    model_name = state["tuning_target_model"]
    params = state["current_params"]

    sig = param_signature(params)
    already_tried = set(state.get("tried_param_signatures", []))

    if sig in already_tried:
        raise ValueError(f"Duplicate parameter combination detected: {params}")

    result = train_and_evaluate_candidate(
        model_name=model_name,
        params=params,
        train_prepared=state["train_prepared"],
        val_prepared=state["val_prepared"],
        test_prepared=state["test_prepared"],
    )

    next_iteration = state["iteration"] + 1

    history = state.get("run_history", []) + [{
        "iteration": next_iteration,
        "params": result["params"],
        "train_rmse": result["train_rmse"],
        "val_rmse": result["val_rmse"],
        "test_rmse": result["test_rmse"],
        "val_r2": result["val_r2"],
        "overfit_gap": result["overfit_gap"],
    }]

    tried_param_signatures = state.get("tried_param_signatures", []) + [sig]
    tried_param_log = state.get("tried_param_log", []) + [{
        "iteration": next_iteration,
        "params": result["params"],
        "signature": sig,
        "val_rmse": result["val_rmse"],
    }]

    prev_best_rmse = state["best_rmse"]
    best = min(history, key=lambda x: x["val_rmse"])
    tuned_name = model_name.replace("_baseline", "_tuned")

    tuned_models = dict(state.get("tuned_models", {}))
    tuned_metrics = dict(state.get("tuned_metrics", {}))

    if result["val_rmse"] < prev_best_rmse:
        tuned_models[tuned_name] = result["fitted_model"]
        tuned_metrics[tuned_name] = result["full_metrics"]

    print(
        f"[Train {next_iteration}] "
        f"val_rmse={result['val_rmse']} "
        f"r2={result['val_r2']} "
        f"gap={result['overfit_gap']}"
    )
    print(f"[Best so far] val_rmse={best['val_rmse']} params={best['params']}")
    print(f"[Tried combos] {len(tried_param_signatures)}")

    return {
        "run_history": history,
        "iteration": next_iteration,
        "best_rmse": best["val_rmse"],
        "best_params": best["params"],
        "tuned_models": tuned_models,
        "tuned_metrics": tuned_metrics,
        "tried_param_signatures": tried_param_signatures,
        "tried_param_log": tried_param_log,
    }

Analyser node

Reviews the results and decides whether tuning should continue or stop.

In [ ]:
def analyser(state: MLAgentState):
    # Guardrail: no training is required
    if state.get("decision") == "done":
        print("Analyser skipped — no training is required")
        return {}

    model_name = state["tuning_target_model"]
    starting_params = state["baseline_hyperparams"][model_name]

    remaining_candidates = generate_candidates(
        model_name=model_name,
        anchor_params=state["best_params"],
        tried_param_signatures=state.get("tried_param_signatures", []),
        max_candidates=8,
    )

    result = tuning_decision(
        model_name=model_name,
        starting_params=starting_params,
        run_history=state["run_history"],
        best_rmse=state["best_rmse"],
        iteration=state["iteration"],
        remaining_candidate_count=len(remaining_candidates),
    )

    print(f"[Analyser] {result['decision']} — {result['reasoning']}")

    return {
        "decision": result["decision"],
        "tuning_reasoning": result["reasoning"],
    }

Router node

Controls the loop flow by sending the workflow back for another tuning round or ending it when the stop conditions are met.

In [ ]:
def router(state: MLAgentState) -> Literal["continue", "done"]:
    if state["decision"] == "done" or state["iteration"] >= 15:
        return "done"
    return "continue"

In [ ]:
graph_builder = StateGraph(MLAgentState)

graph_builder.add_node("planner", planner)
graph_builder.add_node("train", train)
graph_builder.add_node("analyser", analyser)

graph_builder.add_edge(START, "planner")
graph_builder.add_edge("planner", "train")
graph_builder.add_edge("train", "analyser")
graph_builder.add_conditional_edges(
    "analyser",
    router,
    {
        "continue": "planner",
        "done": END,
    },
)

agent = graph_builder.compile()

In [ ]:
state = fine_tuning_agent_init_node(state)

final = agent.invoke(state)

print("\nFine-tuning loop completed.")
print("Model:", final["tuning_target_model"])
print("Best RMSE:", final["best_rmse"])
print("Best params:", final["best_params"])

state = final

[Planner] First run uses baseline params.
[Planner] params={'num_rounds': 200, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'min_child_weight': 1, 'gamma': 0}


INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.8, 'n_workers': 1, 'use_gpu': False, 'num_rounds': 200, 'col_sample_by_tree': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


[Train 1] val_rmse=0.3993 r2=0.5444 gap=0.1639
[Best so far] val_rmse=0.3993 params={'num_rounds': 200, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'min_child_weight': 1, 'gamma': 0}
[Tried combos] 1
[Analyser] continue — Only 1 run completed; not enough to determine stopping criteria. Proceeding to adjust parameters starting with reducing max_depth.
[Planner] Reducing max_depth from 8 to 6 is a reasonable step to potentially address overfitting, as the current overfit_gap is 0.1639. This is the first layer of adjustments according to the tuning logic, targeting a more generalizable model while maintaining other parameters the same.
[Planner] changed max_depth -> 6
[Planner] params={'num_rounds': 200, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'min_child_weight': 1, 'gamma': 0}


INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 1, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.8, 'n_workers': 1, 'use_gpu': False, 'num_rounds': 200, 'col_sample_by_tree': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


[Train 2] val_rmse=0.4068 r2=0.5272 gap=0.1219
[Best so far] val_rmse=0.3993 params={'num_rounds': 200, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0, 'min_child_weight': 1, 'gamma': 0}
[Tried combos] 2
[Analyser] continue — Only 2 runs have completed, and the best val RMSE is currently at 0.3993. There are still untried candidates available, hence we should proceed with further iterations.
[Planner] The current overfit_gap suggests a potential for overfitting, and reducing reg_lambda may help improve generalization without deviating too far from the baseline parameters. Increasing reg_lambda to 0.5 could stabilize the model performance and potentially lower the val RMSE, which is the primary objective. It's a reasonable adjustment given its successful use for regularization in XGBoost.
[Planner] changed reg_lambda -> 0.5
[Planner] params={'num_rounds': 200, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.8, 

...

[Train 13] val_rmse=0.3919 r2=0.5611 gap=0.1687
[Best so far] val_rmse=0.3912 params={'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'gamma': 0}
[Tried combos] 13
[Analyser] continue — The best val RMSE of 0.3912 was achieved on iteration 11, but the improvement across the last three runs was not consistently below 0.002, and there are still remaining candidates to explore. Therefore, we will continue tuning.
[Planner] Candidate 3 changes only 'min_child_weight' back to 1 while maintaining all other promising parameters. This is a sensible option since increasing 'min_child_weight' in previous iterations showed improvement in val RMSE. It balances exploration of parameter space while staying close to the current best parameters.
[Planner] changed min_child_weight -> 1
[Planner] params={'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 1, 'reg_alpha': 0.0, 'reg_lambda': 2.0, 'subsample': 0.8, 'n_workers': 1, 'use_gpu': False, 'num_rounds': 200, 'col_sample_by_tree': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


[Train 14] val_rmse=0.3935 r2=0.5576 gap=0.1951
[Best so far] val_rmse=0.3912 params={'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'gamma': 0}
[Tried combos] 14
[Analyser] continue — At iteration 14, we achieved a best val RMSE of 0.3912, and since we have not yet completed the minimum required 5 runs after achieving this value, we should continue tuning. Furthermore, there are still untried candidates available for exploration.
[Planner] I chose candidate_id 3 because it decreases the reg_lambda from the current best parameters. Given that higher regularization can mitigate overfitting, the previous configurations with reg_lambda at 2.0 showed improvements in val RMSE, especially iterations 11, 12, and 14. Reducing it to 1.0 may help achieve a lower val RMSE, while also considering the overfit_gap which needs to be minimized.
[Planner] changed reg_lambda -> 1.0
[Planne

INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 5, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'subsample': 0.8, 'n_workers': 1, 'use_gpu': False, 'num_rounds': 200, 'col_sample_by_tree': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


[Train 15] val_rmse=0.3952 r2=0.5538 gap=0.1828
[Best so far] val_rmse=0.3912 params={'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'gamma': 0}
[Tried combos] 15
[Analyser] continue — The best val RMSE of 0.3912 has been achieved, but more than 5 runs have been completed and there are still remaining candidate parameters to explore. The recent improvements are small, but it's still beneficial to investigate other candidates to see if a better result can be achieved.

Fine-tuning loop completed.
Model: XGBoost_baseline
Best RMSE: 0.3912
Best params: {'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'gamma': 0}


# 12. Final model selection + evaluation
Purpose:
- Ranks all baseline and tuned models by validation RMSE, MAE, and R².
- Selects the best model and prints a final comparison table.

In [ ]:
def model_display_name(model_name: str) -> str:
    mapping = {
        "LinearRegression_baseline": "LR  Baseline",
        "RandomForest_baseline": "RF  Baseline",
        "XGBoost_baseline": "XGB Baseline",
        "LinearRegression_tuned": "LR  Tuned",
        "RandomForest_tuned": "RF  Tuned",
        "XGBoost_tuned": "XGB Tuned",
    }
    return mapping.get(model_name, model_name)


def final_model_selection_node(state: MLAgentState) -> MLAgentState:
    baseline_metrics = state.get("baseline_metrics", {})
    tuned_metrics = state.get("tuned_metrics", {})

    combined_metrics = {
        **baseline_metrics,
        **tuned_metrics,
    }

    if not combined_metrics:
        raise ValueError("No baseline_metrics or tuned_metrics found in state.")

    # Same ranking rule as baseline evaluation:
    # 1) lowest validation RMSE
    # 2) lower validation MAE
    # 3) higher validation R²
    ranked = sorted(
        combined_metrics.items(),
        key=lambda x: (
            x[1]["val"]["rmse"],
            x[1]["val"]["mae"],
            -x[1]["val"]["r2"],
        ),
    )

    best_model_name, best_metrics = ranked[0]

    state["final_best_model"] = best_model_name
    state["final_ranking"] = [name for name, _ in ranked]
    state["final_evaluation"] = {
        "final_best_model": best_model_name,
        "final_ranking": [name for name, _ in ranked],
        "selection_rule": "Lowest validation RMSE, then lower validation MAE, then higher validation R2",
        "best_val_rmse": best_metrics["val"]["rmse"],
        "best_val_mae": best_metrics["val"]["mae"],
        "best_val_r2": best_metrics["val"]["r2"],
        "best_test_rmse": best_metrics["test"]["rmse"],
        "best_test_mae": best_metrics["test"]["mae"],
        "best_test_r2": best_metrics["test"]["r2"],
    }

    return state


state = final_model_selection_node(state)

# Print best tuned params from the agent
tuned_name = state["tuning_target_model"].replace("_baseline", "_tuned")

if tuned_name in state.get("tuned_metrics", {}):
    print("=" * 70)
    print("BEST TUNED PARAMS")
    print("=" * 70)
    print(f"Model: {model_display_name(tuned_name)}")
    print(state["best_params"])
    print()

# Print final comparison table
all_metrics = {
    **state.get("baseline_metrics", {}),
    **state.get("tuned_metrics", {}),
}

rows = []
for model_name in state["final_ranking"]:
    metrics = all_metrics[model_name]
    rows.append(
        (
            model_display_name(model_name),
            metrics["val"]["rmse"],
            metrics["val"]["r2"],
            metrics["test"]["rmse"],
            metrics["test"]["r2"],
        )
    )

print("=" * 134)
print("FINAL COMPARISON")
print("=" * 134)
print(f"{'Model':<18} {'Val RMSE':>10} {'Val R²':>8} {'Test RMSE':>11} {'Test R²':>9}")
print(f"{'-' * 18} {'-' * 10} {'-' * 8} {'-' * 11} {'-' * 9}")

for model_label, val_rmse, val_r2, test_rmse, test_r2 in rows:
    print(f"{model_label:<18} {val_rmse:>10.4f} {val_r2:>8.4f} {test_rmse:>11.4f} {test_r2:>9.4f}")

print()
print("=" * 70)
print("FINAL SELECTED MODEL")
print("=" * 70)
print(f"Selected model: {model_display_name(state['final_best_model'])}")
print("Selection rule: lowest validation RMSE, then lower validation MAE, then higher validation R²")

best_final_metrics = all_metrics[state["final_best_model"]]
print(
    f"Validation -> RMSE={best_final_metrics['val']['rmse']:.4f}, "
    f"MAE={best_final_metrics['val']['mae']:.4f}, "
    f"R²={best_final_metrics['val']['r2']:.4f}"
)
print(
    f"Test       -> RMSE={best_final_metrics['test']['rmse']:.4f}, "
    f"MAE={best_final_metrics['test']['mae']:.4f}, "
    f"R²={best_final_metrics['test']['r2']:.4f}"
)

BEST TUNED PARAMS
Model: XGB Tuned
{'num_rounds': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'col_sample_by_tree': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.0, 'min_child_weight': 5, 'gamma': 0}

FINAL COMPARISON
Model                Val RMSE   Val R²   Test RMSE   Test R²
------------------ ---------- -------- ----------- ---------
XGB Tuned              0.3912   0.5626      0.2974    0.7146
XGB Baseline           0.3993   0.5444      0.3021    0.7056
LR  Baseline           0.4230   0.4887      0.3411    0.6247
RF  Baseline           0.4799   0.3418      0.4206    0.4293

FINAL SELECTED MODEL
Selected model: XGB Tuned
Selection rule: lowest validation RMSE, then lower validation MAE, then higher validation R²
Validation -> RMSE=0.3912, MAE=0.2282, R²=0.5626
Test       -> RMSE=0.2974, MAE=0.2141, R²=0.7146
